# Calculations and demos of the perturbation theory

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Creating a Fourier grid

In [ ]:
def fourier_grid(nmesh, Lbox, hermitian=False):
    kk = np.fft.fftfreq(nmesh) * 2*np.pi / Lbox * nmesh
    ks = np.fft.rfftfreq(nmesh) * 2*np.pi / Lbox * nmesh if hermitian else kk
    kvec = np.array(np.meshgrid(kk, kk, ks, indexing='ij'))
    kmod = np.linalg.norm(kvec, axis=0)
    return kvec, kmod

In [ ]:
%%timeit -n 100 -r 5
kvec, kmod = fourier_grid(nmesh=128, Lbox=1_000, hermitian=False)

In [ ]:
%%timeit -n 100 -r 5
kvec, kmod = fourier_grid(nmesh=128, Lbox=1_000, hermitian=True)

In [ ]:
kvec, kmod = fourier_grid(nmesh=128, Lbox=1_000, hermitian=False)
print('kvec.sum:              ', kvec.sum())
print('kvec.shape:            ', kvec.shape)
print('kmod.shape:            ', kmod.shape)
kvec, kmod = fourier_grid(nmesh=128, Lbox=1_000, hermitian=True)
print('kvec.sum (Hermitian):  ', kvec.sum())
print('kvec.shape (Hermitian):', kvec.shape)
print('kmod.shape (Hermitian):', kmod.shape)

### Plot wavenumbers

In [ ]:
def plot_wavenumber(ax, nmesh, Lbox):
    kk = np.fft.fftfreq(nmesh) * 2*np.pi / Lbox * nmesh
    ks = np.fft.rfftfreq(nmesh) * 2*np.pi / Lbox * nmesh
    ax.plot(kk, label='k', lw=3)
    ax.plot(ks, label='k [real]', lw=4, ls=':')
    ax.set_xlabel(f'Grid index [0-{nmesh}]')
    ax.set_ylabel('Wavenumber $k$ [h/Mpc]')
    ax.text(0.05, 0.05, f'nmesh = {nmesh}\nLbox = {Lbox} Mpc/h', fontsize=8,
            transform=ax.transAxes, va='bottom', ha='left',
            bbox=dict(boxstyle='square,pad=0.3', ec='black', fc='none', alpha=0.5))
    ax.legend(loc='upper right', fontsize=8)

In [ ]:
nr, nc = 1, 3
fig, axes = plt.subplots(nr, nc, figsize=(nc*5.2, nr*4), dpi=120)
fig.subplots_adjust(wspace=0.3)

plot_wavenumber(axes[0], nmesh=128, Lbox=1_000)
plot_wavenumber(axes[1], nmesh=256, Lbox=2_000)
plot_wavenumber(axes[2], nmesh=384, Lbox=3_000)

plt.show()

#### Fix `nmesh`

In [ ]:
nr, nc = 1, 3
fig, axes = plt.subplots(nr, nc, figsize=(nc*5.2, nr*4), dpi=120)
fig.subplots_adjust(wspace=0.3)

nmesh = 256   # number of mesh points
plot_wavenumber(axes[0], nmesh=nmesh, Lbox=1_000)
plot_wavenumber(axes[1], nmesh=nmesh, Lbox=2_000)
plot_wavenumber(axes[2], nmesh=nmesh, Lbox=3_000)

plt.show()

#### Fix `Lbox`

In [ ]:
nr, nc = 1, 3
fig, axes = plt.subplots(nr, nc, figsize=(nc*5.2, nr*4), dpi=120)
fig.subplots_adjust(wspace=0.3)

Lbox = 1_000  # box size in [Mpc/h]
plot_wavenumber(axes[0], nmesh=128, Lbox=Lbox)
plot_wavenumber(axes[1], nmesh=256, Lbox=Lbox)
plot_wavenumber(axes[2], nmesh=384, Lbox=Lbox)

plt.show()

### Plot wavevectors

In [ ]:
from mpl_toolkits.axes_grid1 import make_axes_locatable

In [ ]:
nmesh = 128   # number of mesh points
Lbox = 1_000  # box size in [Mpc/h]


In [ ]:
kvec, kmod = fourier_grid(nmesh=nmesh, Lbox=Lbox, hermitian=False)

In [ ]:
nr, nc = 1, 2
fig, axes = plt.subplots(nr, nc, figsize=(nc*5.5, nr*4), dpi=120)
fig.subplots_adjust(wspace=-0.7)

slice_idx = kmod.shape[2]//2

step = 4
kx, ky, kz = kvec
kx_s = kx[::step, ::step, slice_idx]
ky_s = ky[::step, ::step, slice_idx]
kmod_s = kmod[::step, ::step, slice_idx]

# quiver of (kx, ky) at fixed kz
ax = axes[0]
ax.quiver(kx_s, ky_s, kx_s/kmod_s, ky_s/kmod_s, kmod_s, # [X, Y], [U, V], [C]
          cmap='plasma', scale=None, width=0.003)
ax.set_title(f'Vector slice at $k_z$ index={slice_idx}',
             loc='left', fontsize=10)
dx, dy = kx_s.max() - kx_s.min(), ky_s.max() - ky_s.min()
x_min, x_max = kx_s.min()-0.05*dx, kx_s.max()+0.05*dx
y_min, y_max = ky_s.min()-0.05*dy, ky_s.max()+0.05*dy
ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)
ax.set_aspect('equal')

# scalar contour of kmod
ax = axes[1]
levels = np.linspace(0.3, 0.5, 100, endpoint=True)
c = ax.contourf(kx_s, ky_s, kmod_s, levels=levels)
ax.set_title(f'Scalar slice of $|\\mathbf{{k}}|$ at $k_z$ index={slice_idx}',
             loc='left', fontsize=10)
ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)
ax.set_aspect('equal')

divider = make_axes_locatable(ax)
cax = divider.append_axes('right', size='5%', pad=0.05)
cb = plt.colorbar(c, cax=cax)
cb.set_ticks(np.linspace(0.3, 0.5, 5))
cb.set_label(r'$|\mathbf{k}|$ [h/Mpc]', rotation=270, labelpad=15)

plt.tight_layout()
plt.show()

In [ ]:
def radial_profile(kmod):
    """Calculate radial profile of kmod."""
    ks = kmod.ravel()
    bins = np.linspace(0, ks.max(), 100)
    hist, edges = np.histogram(ks, bins=bins)
    centers = 0.5*(edges[:-1]+edges[1:])
    return centers, hist

In [ ]:
def grid_info(ax, nmesh, Lbox):
    info = f'nmesh = {nmesh}\nLbox = {Lbox} Mpc/h'
    text = f'\nMin. res. = {2*np.pi/kmod.max():.2f} Mpc/h'
    ax.text(0.025, 0.875, info+text, fontsize=8,
            transform=ax.transAxes, va='top', ha='left',
            bbox=dict(boxstyle='square,pad=0.3', ec='black', fc='none', alpha=0.5))

In [ ]:
def plot_nyquist(ax, nmesh, Lbox):
    k_nyquist = np.pi * nmesh / Lbox
    ax.axvline(x=k_nyquist, color='r', linestyle=':', label='Nyquist frequency')

In [ ]:
nr, nc = 1, 2
fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120, sharey=True)
fig.subplots_adjust(wspace=0.05)

ax = axes[0]
kvec, kmod = fourier_grid(nmesh, Lbox, hermitian=False)
centers, hist = radial_profile(kmod)
ax.step(centers, hist, where='mid')
plot_nyquist(ax, nmesh, Lbox)
# ax.set_yscale('log')
ax.set_xlabel(r'|$\mathbf{k}$|')
ax.set_ylabel('Number of $k$-modes')
ax.set_title('Radial profile of non-Hermitian $k$-modes', loc='left', fontsize=10)
grid_info(ax, nmesh, Lbox)
ax.legend(loc='upper left', fontsize=8)

ax = axes[1]
kvec, kmod = fourier_grid(nmesh, Lbox, hermitian=True)
centers, hist = radial_profile(kmod)
ax.step(centers, hist, where='mid')
plot_nyquist(ax, nmesh, Lbox)
# ax.set_yscale('log')
ax.set_xlabel(r'|$\mathbf{k}$|')
ax.set_title('Radial profile of Hermitian $k$-modes', loc='left', fontsize=10)
grid_info(ax, nmesh, Lbox)
ax.legend(loc='upper left', fontsize=8)

plt.show()

In [ ]:
from skimage import measure
from mpl_toolkits.mplot3d import Axes3D

In [ ]:
def visualize_kmod_isosurfaces(ax, kmod, levels=[0.1, 0.3, 0.5], title=None):
    kmod_norm = kmod / kmod.max()
    
    # Create isosurfaces at different levels
    for level in levels:
        verts, faces, _, _ = measure.marching_cubes(kmod_norm, level=level)
        
        v = (verts[:, 0], verts[:, 1], faces, verts[:, 2])
        ax.plot_trisurf(*v, label=f'k={level:.1f}', alpha=0.3)
    
    ax.set_xlabel('kx')
    ax.set_ylabel('ky')
    ax.set_zlabel('kz')
    ax.set_title(title, fontsize=10)

In [ ]:
nr, nc = 1, 2
fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*6), dpi=120,
                         subplot_kw={'projection': '3d'})
for ax in axes.flat:
    ax.set_box_aspect(None, zoom=0.85)
    ax.view_init(elev=45, azim=35)

kvec, kmod = fourier_grid(nmesh, Lbox, hermitian=False)
levels = [0.1, 0.3, 0.5]
visualize_kmod_isosurfaces(ax=axes[0], kmod=kmod, levels=levels,
                           title='Isosurfaces of non-Hermitian $k$-modes')

kvec, kmod = fourier_grid(nmesh, Lbox, hermitian=True)
levels = [0.1, 0.3, 0.5]
visualize_kmod_isosurfaces(ax=axes[1], kmod=kmod, levels=levels,
                           title='Isosurfaces of Hermitian $k$-modes')

fig.tight_layout()
plt.show()